# 1. Simulate: conditioning, beta tuning and the two openCARP runs

* **preflight**: software versions, repository commit, solver flavour, R availability.
* **conditioning**: one cell paced 1,000 times with `bench` (limit cycle).
* **tune**: beta for the CV target on a strip at the production mesh (FM-18); uses `config/tuning.json`
  if `python -m efov tune --write-par` was run for this mesh and target.
* **simulation**: `efov/opencarp/slab_experiment.py` (carputils) for the intact and lesion slab. Every
  openCARP launch reads one resolved parameter file (`simulations/<condition>/prepace.par`, `run.par`).

Long runs are safer from the terminal: `nohup python -m efov run --out runs/v4_reference --np 24 --until simulation > runs/sim.log 2>&1 &`.

In [ ]:
import os, sys
from pathlib import Path
REPOSITORY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPOSITORY))

# ---- settings: edit here, or set EFOV_CONFIG / EFOV_OUT / EFOV_NP ----
CONFIG = os.environ.get("EFOV_CONFIG", str(REPOSITORY / "config" / "model.json"))
OUT    = os.environ.get("EFOV_OUT",    str(REPOSITORY / "runs" / "v4_reference"))
NP     = int(os.environ.get("EFOV_NP", "24"))      # MPI processes for openCARP
# openCARP's own MPI launcher (efov-runner image); needed when NP > 1
os.environ.setdefault("MPIEXEC_BIN", "/usr/local/lib/opencarp/lib/petsc/bin/mpiexec")

import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from efov.config import loadConfig
from efov.pipeline import Run

config = loadConfig(CONFIG)
run = Run(config, OUT, processCount=NP)
print(f"{config['modelName']} -> {run.out}")
run.status()

In [ ]:
run.runStage("preflight")

In [ ]:
run.runStage("conditioning")

In [ ]:
run.runStage("tune")

In [ ]:
result = run.runStage("simulation")
pd.DataFrame(result["timing"]).T.join(pd.DataFrame(result["nodes"]).T)

The resolved parameter file of the lesion run: model.par with this run's overrides in place, then the run-specific paths.

In [ ]:
print((run.jobs["lesion"] / "run.par").read_text()[-3000:])